# 05 - Feature engineering & preprocessing

Codifica las variables categoricas con un `OneHotEncoder` **ajustado unicamente con
`train_raw`** (de `04_train_test_split.ipynb`) y aplica esa misma transformacion a `test` y a
los clientes `Joined`. A diferencia de `pd.get_dummies` sobre el dataset completo, esto
garantiza que ninguna categoria vista solo en test (o en Joined) genere una columna que el
modelo no pudo aprender a usar, y que el conjunto de columnas de salida sea identico en los
tres conjuntos.

In [1]:
import os

import pandas as pd
from sklearn.preprocessing import OneHotEncoder

PROCESSED_DIR = "../data/processed"
CUSTOMER_ID_COL = "Customer_ID"

train_raw = pd.read_parquet(f"{PROCESSED_DIR}/train_raw.parquet")
test_raw = pd.read_parquet(f"{PROCESSED_DIR}/test_raw.parquet")
joined_raw = pd.read_parquet(f"{PROCESSED_DIR}/joined_raw.parquet")

X_train = train_raw.drop(columns=["churn_flag"])
y_train = train_raw["churn_flag"]
X_test = test_raw.drop(columns=["churn_flag"])
y_test = test_raw["churn_flag"]

joined_ids = joined_raw[CUSTOMER_ID_COL].reset_index(drop=True)
X_joined = joined_raw.drop(columns=[CUSTOMER_ID_COL])

categorical_cols = X_train.select_dtypes(include=["object", "str"]).columns.tolist()
numeric_cols = [c for c in X_train.columns if c not in categorical_cols]
print(f"Columnas categoricas ({len(categorical_cols)}): {categorical_cols}")
print(f"Columnas numericas ({len(numeric_cols)}): {numeric_cols}")

Columnas categoricas (19): ['Gender', 'Married', 'State', 'Value_Deal', 'Phone_Service', 'Multiple_Lines', 'Internet_Service', 'Internet_Type', 'Online_Security', 'Online_Backup', 'Device_Protection_Plan', 'Premium_Support', 'Streaming_TV', 'Streaming_Movies', 'Streaming_Music', 'Unlimited_Data', 'Contract', 'Paperless_Billing', 'Payment_Method']
Columnas numericas (4): ['Age', 'Number_of_Referrals', 'Tenure_in_Months', 'Monthly_Charge']


## Ajustar el encoder solo con `X_train`

`handle_unknown="ignore"` evita que una categoria nueva en test/Joined (que el encoder nunca
vio en train) rompa la transformacion -- simplemente queda en 0 en todas las columnas
one-hot de esa variable, en vez de generar una columna nueva o lanzar un error.

In [2]:
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype="int")
encoder.fit(X_train[categorical_cols])

encoded_cols = encoder.get_feature_names_out(categorical_cols)

def encode(frame):
    encoded = pd.DataFrame(
        encoder.transform(frame[categorical_cols]), columns=encoded_cols, index=frame.index
    )
    return pd.concat([frame[numeric_cols].reset_index(drop=True), encoded.reset_index(drop=True)], axis=1)

X_train_encoded = encode(X_train)
X_test_encoded = encode(X_test)
X_joined_encoded = encode(X_joined)

print(f"X_train_encoded: {X_train_encoded.shape}")
print(f"X_test_encoded:  {X_test_encoded.shape}")
print(f"X_joined_encoded: {X_joined_encoded.shape}")
assert list(X_train_encoded.columns) == list(X_test_encoded.columns) == list(X_joined_encoded.columns)

X_train_encoded: (4724, 79)
X_test_encoded:  (1182, 79)
X_joined_encoded: (405, 79)


## Guardar los conjuntos codificados

In [3]:
os.makedirs(PROCESSED_DIR, exist_ok=True)

train_encoded = X_train_encoded.copy()
train_encoded["churn_flag"] = y_train.reset_index(drop=True)
train_encoded.to_parquet(f"{PROCESSED_DIR}/train_encoded.parquet", index=False)

test_encoded = X_test_encoded.copy()
test_encoded["churn_flag"] = y_test.reset_index(drop=True)
test_encoded.to_parquet(f"{PROCESSED_DIR}/test_encoded.parquet", index=False)

joined_encoded = X_joined_encoded.copy()
joined_encoded.insert(0, "Customer_ID", joined_ids.values)
joined_encoded.to_parquet(f"{PROCESSED_DIR}/joined_encoded.parquet", index=False)

print("Guardado: train_encoded.parquet, test_encoded.parquet, joined_encoded.parquet")

Guardado: train_encoded.parquet, test_encoded.parquet, joined_encoded.parquet
